# 02 — Data Cleaning & Feature Engineering
Transforms raw match data into a model-ready dataset: results, points, gameweek, title gap, high-stakes flags, Drop Index, rolling features, and recency weights.

## Imports

In [2]:
import pandas as pd
import numpy as np
import soccerdata as sd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print("\n✅ Imports ready")

pandas : 3.0.5
numpy  : 2.4.6

✅ Imports ready


## Paths & Config

In [3]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROC_DATA_DIR.mkdir(parents=True, exist_ok = True)

TITLE_TEAMS = ['Arsenal','Liverpool','Manchester City','Manchester United']
PL = "ENG-Premier League"
ARTETA_SEASONS = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

# Recency weights per season — 25-26 counts 1.5x in the ML model
SEASON_WEIGHTS = {
    '1920': 1.0, '2021': 1.0, '2122': 1.0,
    '2223': 1.1, '2324': 1.2, '2425': 1.3,
    '2526': 1.5,
}

print(f"Raw data dir       : {RAW_DATA_DIR}")
print(f"Processed data dir : {PROC_DATA_DIR}")
print(f"Season weights     : {SEASON_WEIGHTS}")

Raw data dir       : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\raw
Processed data dir : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\processed
Season weights     : {'1920': 1.0, '2021': 1.0, '2122': 1.0, '2223': 1.1, '2324': 1.2, '2425': 1.3, '2526': 1.5}


## Load and Combine All 4 Teams

In [9]:
# Load all 4 title teams and stack into one DataFrame

dfs = [
    pd.read_csv(RAW_DATA_DIR / f"{team.lower().replace(' ', '_')}_raw.csv")
    for team in TITLE_TEAMS
]

df = pd.concat(dfs, ignore_index = True)

# Fix date column type
df['date'] = pd.to_datetime(df['date'])

# Sort by team then date — critical for rolling features to work correctly
df = df.sort_values(['team','date']).reset_index(drop=True)


print(f"Combined shape : {df.shape}")
print(f"Teams          : {sorted(df['team'].unique())}")
print(f"Date range     : {df['date'].min().date()} → {df['date'].max().date()}")
df.head(5)

Combined shape : (1064, 11)
Teams          : ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']
Date range     : 2019-08-09 → 2026-05-24


,league,season,game_id,date,team,opponent,venue,xG,xGA,scored,conceded
0,ENG-Premier League,1920,11650,2019-08-11 14:00:00,Arsenal,Newcastle United,away,1.133090,0.380551,1,0
1,ENG-Premier League,1920,11653,2019-08-17 12:30:00,Arsenal,Burnley,home,1.164400,1.391720,1,2
2,ENG-Premier League,1920,11670,2019-08-24 17:30:00,Arsenal,Liverpool,away,0.985542,2.788210,1,3
3,ENG-Premier League,1920,11682,2019-09-01 16:30:00,Arsenal,Tottenham,home,1.925090,1.955140,2,2
4,ENG-Premier League,1920,11691,2019-09-15 15:30:00,Arsenal,Watford,away,1.006160,2.832090,2,2


## Compute Result and Points

In [10]:
# Compute result — W/D/L from scored vs conceded

df['result'] = np.where(
    df['scored'] > df['conceded'], 'W',
    np.where(df['scored'] == df['conceded'], 'D', 'L')
)

# Compute points — 3 for win, 1 for draw, 0 for loss
df['points'] = df['result'].map({'W' : 3, 'D' : 1, 'L' : 0})

# Compute goal difference per match
df['gd'] = df['scored'] - df['conceded']

# Compute xG difference
df['xgd'] = df['xG'] - df['xGA']

print("Result distribution across all 4 teams:")
print(df['result'].value_counts())
print()
print("Points distribution:")
print(df['points'].value_counts().sort_index())
print()
print("Spot check — first 5 Arsenal rows:")
df[df['team'] == 'Arsenal'][['date', 'opponent', 'scored', 'conceded', 'result', 'points']].head(5)

Result distribution across all 4 teams:
result
L    491
W    351
D    222
Name: count, dtype: int64

Points distribution:
points
0    491
1    222
3    351
Name: count, dtype: int64

Spot check — first 5 Arsenal rows:


,date,opponent,scored,conceded,result,points
0,2019-08-11 14:00:00,Newcastle United,1,0,W,3
1,2019-08-17 12:30:00,Burnley,1,2,L,0
2,2019-08-24 17:30:00,Liverpool,1,3,L,0
3,2019-09-01 16:30:00,Tottenham,2,2,D,1
4,2019-09-15 15:30:00,Watford,2,2,D,1


## Add Gameweek

In [11]:
# Compute gameweek: rank matches chronologically within each team-season

df['gameweek'] = (
    df.groupby(['team','season'])['date']
    .rank(method = 'dense')
    .astype(int)
)

# Verify — each team in each season should have gameweeks 1 to 38
gw_check = df.groupby(['team', 'season'])['gameweek'].max().unstack(fill_value=0)
print("Max gameweek per team per season (should all be 38):")
print(gw_check)

Max gameweek per team per season (should all be 38):
season             1920  2021  2122  2223  2324  2425  2526
team                                                       
Arsenal              38    38    38    38    38    38    38
Liverpool            38    38    38    38    38    38    38
Manchester City      38    38    38    38    38    38    38
Manchester United    38    38    38    38    38    38    38


## Reconstruct Full PL Title Table

## Compute Title Gap

## Merge Title Gap into Main DataFrame

## Define High-Stakes Matches

## Build the Drop Index

## Rolling Features
**Critical:** always `.shift(1)` before `.rolling()` — no exceptions.

## Recency Weights

## Sanity Checks

## Save to Processed